# 03 Model 1: Word Embeddings + Classifier
Training of architecture 1: frozen pretrained word embeddings with a 2-layer classifier (ReLU non-linearity). Only the classifier is trained. Experiments are tracked with Weights & Biases.

**In short.** Question: how large must the classifier be, and which learning rate and batch size work best with the fixed-position input? Answer: all candidate runs that learned are close. Chosen: 100 hidden units, learning rate 0.000316, batch size 32, best epoch 13 (validation loss 0.6445, accuracy 63.20%).

Load the features and labels that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only evaluated once at the end, in `05_evaluation.ipynb`.

In [1]:
import numpy as np
import torch

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)

torch.Size([16801, 1200]) torch.float32
torch.Size([16801]) torch.int64


## Classifier
The 2-layer classifier like in the course exercise "PyTorch" (Week 1): linear layer, ReLU, linear layer. Input: 1200 features, output: 2 classes (h1 or h2).

In [2]:
import torch.nn as nn

class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
print(model)

Classifier(
  (linear1): Linear(in_features=1200, out_features=100, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=100, out_features=2, bias=True)
)


Number of parameters per layer (weights and biases), as in the course exercise "PyTorch" (Week 1).

In [3]:
for name, params in model.named_parameters():
    print(name, params.shape)

linear1.weight torch.Size([100, 1200])
linear1.bias torch.Size([100])
linear2.weight torch.Size([2, 100])
linear2.bias torch.Size([2])


Test the untrained classifier on the first 5 training examples: one pair of logits (h1, h2) per example.

In [4]:
x = X_train[:5]
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 1200])
torch.Size([5, 2])
tensor([[-0.1848, -0.1143],
        [-0.1771, -0.0636],
        [-0.1528, -0.0544],
        [-0.1209, -0.0621],
        [-0.1661, -0.0300]], grad_fn=<AddmmBackward0>)


## Loss
Cross-entropy loss for the 2 classes (course slides "Preliminaries", slide 27). `nn.CrossEntropyLoss` applies the softmax to the logits itself, so the classifier has no softmax layer. A binary decision can be made with 1 output and a sigmoid or with 2 outputs and a softmax (course slides "Project Discussions" (Week 5), slide 22). 2 outputs are used, one logit per hypothesis, as in the classifier of the course exercise "PyTorch", so the prediction is the hypothesis with the higher logit (`argmax`). A softmax layer in the model would apply the softmax twice (the "double sigmoid" warning on slide 30 of the same slides, Week 6). Before training, the loss should be about -log(0.5) ≈ 0.69, the value of random guessing between 2 classes.

In [5]:
criterion = nn.CrossEntropyLoss()

y = y_train[:5]
loss = criterion(output, y)
print(y)
print(loss)

tensor([1, 1, 0, 0, 1])
tensor(0.6781, grad_fn=<NllLossBackward0>)


## One training step 
The Adam optimizer updates the weights (course slides "Preliminaries", slide 29). Adam is used rather than AdamW (course slides "Project Discussions" (Week 6), slide 31): AdamW only changes how weight decay is applied, and no weight decay is used here, so both would make the same update steps (algorithm in the PyTorch documentation `torch.optim.AdamW`; Loshchilov & Hutter, 2019). One training step on the 5 examples as in the course exercise "PyTorch" (Week 1): forward pass, loss, backpropagation, update. Afterwards, the loss on the same 5 examples should be lower.

In [6]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=0.001)

output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(loss)
print(criterion(model(x), y))

tensor(0.6781, grad_fn=<NllLossBackward0>)
tensor(0.6114, grad_fn=<NllLossBackward0>)


## Batches
The data is split into batches with a `DataLoader`, as in the course exercise "Experiment Tracking": training data in batches of 32, shuffled in every epoch. The course recommends the largest batch size that fits into the GPU memory (course slides "Project Discussions" (Week 4), slide 14); here the training runs on the CPU, where memory is not the limit, and one epoch takes only a few seconds, so the batch size of the course exercise is kept (the section "Sweep" also tests batch size 128). 16801 = 525 x 32 + 1, so the last batch would contain a single example; `drop_last=True` skips it (a different example in every epoch, because of the shuffling). For measuring loss and accuracy, all batches have the same size, so that the average of the batch losses equals the average over all examples: validation data in 10 batches of 100, training data in 53 batches of 317 (16801 = 53 x 317), both not shuffled. `TensorDataset` pairs each row of features with its label.

In [7]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

525 53 10
torch.Size([32, 1200]) torch.Size([32])


## Evaluation
Loss and accuracy on a dataset, batch by batch, as in the validation of the course exercise "Experiment Tracking": the loss of each batch and the predictions (`argmax`) and labels are collected; at the end, the loss is the average of the batch losses and the accuracy is the share of correct predictions over all examples. `model.eval()` and `torch.no_grad()` as in the `test_loop` of the PyTorch tutorial "Optimizing Model Parameters". A new untrained model is tested first: it should be at about 0.69 loss and 50% accuracy.

In [8]:
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6943, accuracy: 49.10%


## Experiment tracking
The training runs are tracked with Weights and Biases (W&B), as in the course exercise "Experiment Tracking". Log in to W&B (the API key was saved once with `wandb login` in the terminal). `WANDB_SILENT` keeps the log lines of W&B out of the notebook output; the metrics are still logged.

In [9]:
import os

# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import wandb
wandb.login()


True

## Training
`train()` follows the PyTorch tutorial "Optimizing Model Parameters" and the course exercise "Experiment Tracking". Every call trains a fresh model with seed 42, creates one W&B run and logs the training and validation loss and accuracy after every epoch. The cell ends with a walk-through run (10 epochs).

- **Checkpointing:** the `state_dict` of the epoch with the lowest validation loss is saved and returned. The validation loss changes more smoothly than the accuracy. The best values go into the W&B run summary, so that the runs can be sorted by them.
- **Early stopping:** a run stops after 10 epochs without a lower validation loss (patience 10, course slides "Project Discussions" (Week 6), slide 38), because the best epoch comes long before the planned maximum. The validation loss is noisy: with 100 hidden units and learning rate 0.01, it is 0.6613 in epoch 12 and only lower again in epoch 18, so a patience of 5 would stop too early.
- **Learning rate:** constant, the simplest choice, which can also work well (same slides, slide 33, with Hägele et al., 2024).
- **Regularization:** no dropout or weight decay (same slides, slide 34): selecting the best epoch already keeps the model from before it overfits.
- **Normalization:** none in the chosen configuration (same slide); tested in the section "Layer normalization".

In [10]:
import os

os.makedirs("checkpoints", exist_ok=True)

def train(inner_dim=100, learning_rate=0.001, num_epochs=10, patience=10):
    name = f"glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art-final",
        name=name,
        config={
            "model": "glove-mlp",
            "inner_dim": inner_dim,
            "learning_rate": learning_rate,
            "max_epochs": num_epochs,
            "patience": patience,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    torch.manual_seed(42)
    model = Classifier(1200, 2, inner_dim=inner_dim)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                print(f"Early stopping: no lower validation loss in the last {patience} epochs")
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

model = train(inner_dim=100, learning_rate=0.001)

Epoch 1: train loss 0.6693, train acc 60.63%, valid loss 0.6751, valid acc 58.70%
Epoch 2: train loss 0.6449, train acc 62.09%, valid loss 0.6654, valid acc 57.30%
Epoch 3: train loss 0.6251, train acc 64.97%, valid loss 0.6576, valid acc 60.50%
Epoch 4: train loss 0.6187, train acc 65.10%, valid loss 0.6590, valid acc 59.60%
Epoch 5: train loss 0.6051, train acc 66.71%, valid loss 0.6511, valid acc 61.50%
Epoch 6: train loss 0.6058, train acc 66.32%, valid loss 0.6654, valid acc 59.90%
Epoch 7: train loss 0.5844, train acc 68.74%, valid loss 0.6530, valid acc 62.20%
Epoch 8: train loss 0.5737, train acc 69.86%, valid loss 0.6494, valid acc 62.70%
Epoch 9: train loss 0.5630, train acc 70.32%, valid loss 0.6572, valid acc 60.20%
Epoch 10: train loss 0.5458, train acc 72.52%, valid loss 0.6500, valid acc 63.30%
Best epoch: 8, valid loss 0.6494, valid acc 62.70%


Check that the returned model is the one of the best epoch: its validation loss and accuracy must be the same as in that epoch.

In [11]:
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6494, accuracy: 62.70%


## Experiments
Hyperparameter tuning on the validation set (course slides "Course Project HS26", slide 10: the validation set is used to compare model settings and to select the best model and hyperparameters). The experiments answer two questions: how large does the classifier have to be (number of hidden units), and which learning rate works, "the most important hyperparameter" (course slides "Project Discussions" (Week 6), slide 32)? The number of hidden units of the classifier (50, 100, 300) and the learning rate (0.001, 0.01) are varied in a grid search (all 6 combinations, course slides "Project Discussions" (Week 7), slide 44), each for at most 20 epochs. In every run, the best epoch is selected by the validation loss. Because the seed is set inside `train()`, the two runs with the same number of hidden units start from the same initial weights and see the batches in the same order, so the difference between them comes only from the learning rate. Each configuration is trained once. With seed 42, a run is exactly reproducible (the first 10 epochs of the 10-epoch run above and of the 20-epoch run with the same settings are identical), but the differences between configurations can partly be chance; the section "Sweep" compares runs example by example with the exact McNemar test.

In [12]:
for inner_dim in [50, 100, 300]:
    for learning_rate in [0.001, 0.01]:
        train(inner_dim=inner_dim, learning_rate=learning_rate, num_epochs=20)

Epoch 1: train loss 0.6716, train acc 59.84%, valid loss 0.6774, valid acc 58.10%
Epoch 2: train loss 0.6465, train acc 63.16%, valid loss 0.6638, valid acc 58.20%
Epoch 3: train loss 0.6361, train acc 63.41%, valid loss 0.6631, valid acc 59.70%
Epoch 4: train loss 0.6242, train acc 65.13%, valid loss 0.6533, valid acc 61.00%
Epoch 5: train loss 0.6180, train acc 65.81%, valid loss 0.6509, valid acc 60.40%
Epoch 6: train loss 0.6158, train acc 65.93%, valid loss 0.6554, valid acc 60.10%
Epoch 7: train loss 0.6127, train acc 66.07%, valid loss 0.6527, valid acc 61.00%
Epoch 8: train loss 0.6175, train acc 65.13%, valid loss 0.6656, valid acc 60.40%
Epoch 9: train loss 0.6164, train acc 65.08%, valid loss 0.6538, valid acc 61.90%
Epoch 10: train loss 0.6096, train acc 66.24%, valid loss 0.6616, valid acc 60.10%
Epoch 11: train loss 0.6034, train acc 66.74%, valid loss 0.6574, valid acc 60.20%
Epoch 12: train loss 0.6001, train acc 66.73%, valid loss 0.6554, valid acc 62.20%
Epoch 13: tra

## Learning rate 0.0001
The best learning rate of the grid, 0.001, is its smallest value, on the boundary of the tested range. Following the course slides "Project Discussions" (Week 7), slide 53 ("Don't stop testing hyperparameter values when the best performance has a value on the boundary of your testing range"), the next smaller value in exponential steps (slide 52), 0.0001, is tested too, for 50, 100 and 300 hidden units. With 10 times smaller update steps, the model needs more epochs, so these runs have at most 100 epochs instead of 20 (slide 41: select a number of epochs that makes sense).

In [13]:
# the next smaller learning rate, with more epochs (smaller steps)
for inner_dim in [50, 100, 300]:
    train(inner_dim=inner_dim, learning_rate=0.0001, num_epochs=100)

Epoch 1: train loss 0.6862, train acc 53.65%, valid loss 0.6877, valid acc 53.70%
Epoch 2: train loss 0.6741, train acc 61.29%, valid loss 0.6792, valid acc 58.70%
Epoch 3: train loss 0.6628, train acc 62.04%, valid loss 0.6719, valid acc 57.00%
Epoch 4: train loss 0.6541, train acc 62.52%, valid loss 0.6667, valid acc 57.30%
Epoch 5: train loss 0.6454, train acc 63.57%, valid loss 0.6620, valid acc 59.60%
Epoch 6: train loss 0.6387, train acc 64.05%, valid loss 0.6599, valid acc 60.40%
Epoch 7: train loss 0.6335, train acc 64.63%, valid loss 0.6564, valid acc 59.30%
Epoch 8: train loss 0.6286, train acc 65.02%, valid loss 0.6569, valid acc 61.10%
Epoch 9: train loss 0.6238, train acc 65.51%, valid loss 0.6545, valid acc 60.70%
Epoch 10: train loss 0.6206, train acc 65.89%, valid loss 0.6544, valid acc 60.00%
Epoch 11: train loss 0.6162, train acc 66.07%, valid loss 0.6533, valid acc 61.30%
Epoch 12: train loss 0.6128, train acc 66.54%, valid loss 0.6529, valid acc 61.00%
Epoch 13: tra

## Extending the tested range: learning rate 0.00001 and 1000 hidden units
The best run so far, 300 hidden units with learning rate 0.0001, is on the boundary of the tested range twice: it has the smallest learning rate and the most hidden units. Following slide 53 of the course slides "Project Discussions" (Week 7), the next value is tested in both directions, each with the other setting of the best run: learning rate 0.00001, the next exponential step (slide 52), with at most 300 epochs because the update steps are 10 times smaller, and 1000 hidden units, about 3 times more as from 100 to 300, with at most 100 epochs.

In the grid, the run with 100 hidden units and learning rate 0.01 has its best epoch at the last of its 20 epochs, with the validation loss still falling, so this configuration is also trained for at most 100 epochs (slide 41: select a number of epochs that makes sense). The seed is set inside `train()`, so its first 20 epochs are the same as in the grid.

In [14]:
# test the next values: a smaller learning rate and more hidden units;
# and more epochs for the run whose best epoch was its last epoch
for inner_dim, learning_rate, num_epochs in [(300, 0.00001, 300), (1000, 0.0001, 100), (100, 0.01, 100)]:
    train(inner_dim=inner_dim, learning_rate=learning_rate, num_epochs=num_epochs)

Epoch 1: train loss 0.6912, train acc 53.63%, valid loss 0.6918, valid acc 53.40%
Epoch 2: train loss 0.6893, train acc 56.96%, valid loss 0.6906, valid acc 54.80%
Epoch 3: train loss 0.6871, train acc 58.41%, valid loss 0.6888, valid acc 55.20%
Epoch 4: train loss 0.6849, train acc 58.49%, valid loss 0.6871, valid acc 55.50%
Epoch 5: train loss 0.6825, train acc 60.31%, valid loss 0.6853, valid acc 57.30%
Epoch 6: train loss 0.6801, train acc 60.82%, valid loss 0.6836, valid acc 56.60%
Epoch 7: train loss 0.6777, train acc 60.76%, valid loss 0.6821, valid acc 58.90%
Epoch 8: train loss 0.6757, train acc 59.63%, valid loss 0.6804, valid acc 55.80%
Epoch 9: train loss 0.6734, train acc 60.34%, valid loss 0.6796, valid acc 59.50%
Epoch 10: train loss 0.6704, train acc 61.67%, valid loss 0.6769, valid acc 59.60%
Epoch 11: train loss 0.6680, train acc 62.07%, valid loss 0.6753, valid acc 58.40%
Epoch 12: train loss 0.6658, train acc 62.18%, valid loss 0.6739, valid acc 58.40%
Epoch 13: tra

## Layer normalization
Does layer normalization lower the validation loss of the best configuration of the grid? The course lists normalization as an option (course slides "Project Discussions" (Week 6), slide 34) and layer norm yes/no as an example of an experiment (same slides (Week 7), slide 44). This configuration (300 hidden units, learning rate 0.0001, at most 100 epochs) is trained once more with `nn.LayerNorm` between the first linear layer and the ReLU, because Ba et al. (2016) apply it before the nonlinearity. The check below shows that both runs start from the same weights and get the same first batch, so any difference comes from the layer norm; one run cannot show whether a small difference is more than chance. As for all runs, the best epoch is selected by the validation loss, and if this run has the lowest validation loss of all candidate runs, it becomes the chosen configuration. The training steps are the same as in `train()`; only the class differs. Every 10th epoch is printed; all epochs are logged to W&B.

In [15]:
# the same classifier, with layer normalization between the first linear layer and the ReLU
class LayerNormClassifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        # new: normalizes the inner_dim values of each example
        self.norm = nn.LayerNorm(inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        # [batch, inner_dim]
        out = self.linear1(x)
        # new: layer normalization, same shape
        out = self.norm(out)
        # ReLU, same shape
        out = self.relu(out)
        # [batch, 2]: one logit per hypothesis
        out = self.linear2(out)
        return out


In [16]:
# same seed as in train(): the model with layer norm, then the first batch
torch.manual_seed(42)
model_ln = LayerNormClassifier(1200, 2, inner_dim=300)
x_ln, _ = next(iter(train_loader))

# the same for the model without layer norm
torch.manual_seed(42)
model_plain = Classifier(1200, 2, inner_dim=300)
x_plain, _ = next(iter(train_loader))

# same starting weights and same first batch?
print(torch.equal(model_ln.linear1.weight, model_plain.linear1.weight))
print(torch.equal(model_ln.linear2.weight, model_plain.linear2.weight))
print(torch.equal(x_ln, x_plain))
# how many parameters does layer norm add?
print(sum(p.numel() for p in model_ln.parameters()) - sum(p.numel() for p in model_plain.parameters()))


True
True
True
600


In [17]:
# the best configuration of the grid, with layer norm
inner_dim, learning_rate, num_epochs, patience = 300, 0.0001, 100, 10
name = f"glove-mlp-ln-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-final",
    name=name,
    config={
        "model": "glove-mlp-ln",
        "inner_dim": inner_dim,
        "learning_rate": learning_rate,
        "max_epochs": num_epochs,
        "patience": patience,
        "batch_size": train_loader.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with the layer norm class
torch.manual_seed(42)
model = LayerNormClassifier(1200, 2, inner_dim=inner_dim)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)
    # print every 10th epoch (all epochs are in W&B)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        # early stopping: no new lowest validation loss for `patience` epochs in a row
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print(f"Early stopping: no lower validation loss in the last {patience} epochs")
            break

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, without layer norm (best run of the grid): best epoch 14, valid loss 0.6479, valid acc 62.90%")

Epoch 10: train loss 0.3303, train acc 88.42%, valid loss 0.7270, valid acc 60.00%
Early stopping: no lower validation loss in the last 10 epochs
Best epoch: 3, valid loss 0.6554, valid acc 61.40%
For comparison, without layer norm (best run of the grid): best epoch 14, valid loss 0.6479, valid acc 62.90%


## Results
Load the best checkpoint of every run so far (the sweep follows in its own section), evaluate it on the validation set, and sort the results by the validation loss (lowest first).

In [18]:
results = []
# all runs: hidden units, learning rate and number of epochs
runs = [(inner_dim, learning_rate, num_epochs) for inner_dim in [50, 100, 300]
        for learning_rate, num_epochs in [(0.0001, 100), (0.001, 20), (0.01, 20)]]
runs += [(300, 0.00001, 300), (1000, 0.0001, 100), (100, 0.01, 100)]

for inner_dim, learning_rate, num_epochs in runs:
    model = Classifier(1200, 2, inner_dim=inner_dim)
    model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}.pth", weights_only=True))
    valid_loss, valid_acc = evaluate(model, valid_loader)
    results.append((valid_loss, valid_acc, inner_dim, learning_rate, num_epochs))

# the run with layer norm: the same evaluation, with its own class
model = LayerNormClassifier(1200, 2, inner_dim=300)
model.load_state_dict(torch.load("checkpoints/glove-mlp-ln-h300-lr0.0001-e100.pth", weights_only=True))
valid_loss, valid_acc = evaluate(model, valid_loader)
results.append((valid_loss, valid_acc, "300 with layer norm", 0.0001, 100))

for valid_loss, valid_acc, inner_dim, learning_rate, num_epochs in sorted(results):
    print(f"h{inner_dim}, lr {learning_rate}, {num_epochs} epochs: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")

h300, lr 0.0001, 100 epochs: valid loss 0.6479, valid acc 62.90%
h100, lr 0.0001, 100 epochs: valid loss 0.6489, valid acc 62.60%
h100, lr 0.001, 20 epochs: valid loss 0.6494, valid acc 62.70%
h1000, lr 0.0001, 100 epochs: valid loss 0.6495, valid acc 61.90%
h300, lr 1e-05, 300 epochs: valid loss 0.6500, valid acc 62.10%
h50, lr 0.0001, 100 epochs: valid loss 0.6505, valid acc 62.10%
h50, lr 0.001, 20 epochs: valid loss 0.6505, valid acc 61.90%
h100, lr 0.01, 20 epochs: valid loss 0.6524, valid acc 62.00%
h100, lr 0.01, 100 epochs: valid loss 0.6524, valid acc 62.00%
h300, lr 0.01, 20 epochs: valid loss 0.6540, valid acc 62.70%
h300, lr 0.001, 20 epochs: valid loss 0.6550, valid acc 60.20%
h300 with layer norm, lr 0.0001, 100 epochs: valid loss 0.6554, valid acc 61.40%
h50, lr 0.01, 20 epochs: valid loss 0.6930, valid acc 50.80%


## Active ReLU units
With learning rate 0.01 and 50 hidden units, the model predicts the same class for every example. Count how many hidden units are active, i.e. output a value greater than 0 after the ReLU for more than 1% of the training examples (169 or more of the 16801): before training, and in the best checkpoints of the nine runs with 50, 100, 300 hidden units and learning rates 0.01, 0.001, 0.0001. The 1% is a practical threshold, not the definition of a dead unit: a unit that outputs 0 for every example gets no gradient and no longer learns ("dying ReLU", Lu et al., 2020), while a rarely active unit can still learn. The untrained models are created with seed 42, so they have the same start weights as the training runs with the same number of hidden units.

In [19]:
# no training here, so no gradients are needed
with torch.no_grad():
    for inner_dim in [50, 100, 300]:
        # same start weights as the training runs with this number of hidden units
        torch.manual_seed(42)
        untrained = Classifier(1200, 2, inner_dim=inner_dim)
        # hidden layer only (linear1 + ReLU): one row per example, one column per unit
        hidden = untrained.relu(untrained.linear1(X_train))
        # share of the examples on which each unit outputs more than 0; active if more than 1%
        active = (hidden > 0).float().mean(dim=0) > 0.01
        # count the active units (True counts as 1)
        print(f"h{inner_dim}, untrained: {active.sum().item()} of {inner_dim} units active")
        # the same for the trained models
        for learning_rate, num_epochs in [(0.0001, 100), (0.001, 20), (0.01, 20)]:
            # load the weights of the best epoch of the run (-e20 / -e100 = the run had 20 / 100 epochs)
            model = Classifier(1200, 2, inner_dim=inner_dim)
            model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}.pth", weights_only=True))
            hidden = model.relu(model.linear1(X_train))
            active = (hidden > 0).float().mean(dim=0) > 0.01
            print(f"h{inner_dim}, lr {learning_rate}: {active.sum().item()} of {inner_dim} units active")

h50, untrained: 49 of 50 units active
h50, lr 0.0001: 50 of 50 units active
h50, lr 0.001: 5 of 50 units active
h50, lr 0.01: 0 of 50 units active
h100, untrained: 94 of 100 units active
h100, lr 0.0001: 98 of 100 units active
h100, lr 0.001: 17 of 100 units active
h100, lr 0.01: 1 of 100 units active
h300, untrained: 279 of 300 units active
h300, lr 0.0001: 293 of 300 units active
h300, lr 0.001: 40 of 300 units active
h300, lr 0.01: 1 of 300 units active


## Hypothesis-only check
How much of the result comes from the hypotheses alone? The best configuration of the grid (300 hidden units, learning rate 0.0001, at most 100 epochs) is trained once more with only h1 and h2 as input (the last 600 of the 1200 numbers), without O1 and O2. If it reaches about the same validation loss and accuracy, the observations add little, and most of the result can be reached by judging which hypothesis is more plausible on its own: a hypothesis-only baseline (Gururangan et al., 2018; Poliak et al., 2018; Bhagavatula et al., 2020, Table 1: BERT with the hypothesis only, 59.5% on the ART test set). This run is a diagnostic baseline, not a candidate for model 1: the task is to choose the hypothesis that explains the observations, so the input of model 1 stays all four texts. The training steps are the same as in `train()`; only the input size (600) and the loaders differ. Every 10th epoch is printed; all epochs are logged to W&B.

In [20]:
# only h1 and h2: the last 600 of the 1200 numbers (O1, O2, h1, h2, 300 each)
X_train_hyp = X_train[:, 600:]
X_valid_hyp = X_valid[:, 600:]
# the same batches as for the other runs
train_loader_hyp = DataLoader(TensorDataset(X_train_hyp, y_train), batch_size=32, shuffle=True, drop_last=True)
train_eval_loader_hyp = DataLoader(TensorDataset(X_train_hyp, y_train), batch_size=317)
valid_loader_hyp = DataLoader(TensorDataset(X_valid_hyp, y_valid), batch_size=100)

# the best configuration of the grid
inner_dim, learning_rate, num_epochs, patience = 300, 0.0001, 100, 10
name = f"glove-mlp-hyp-only-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-final",
    name=name,
    config={
        "model": "glove-mlp-hyp-only",
        "inner_dim": inner_dim,
        "learning_rate": learning_rate,
        "max_epochs": num_epochs,
        "patience": patience,
        "batch_size": train_loader_hyp.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with 600 inputs instead of 1200
torch.manual_seed(42)
model = Classifier(600, 2, inner_dim=inner_dim)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader_hyp:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader_hyp)
    valid_loss, valid_acc = evaluate(model, valid_loader_hyp)
    # print every 10th epoch (all epochs are in W&B)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        # early stopping: no new lowest validation loss for `patience` epochs in a row
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print(f"Early stopping: no lower validation loss in the last {patience} epochs")
            break

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, all four texts (best run of the grid): valid loss 0.6479, valid acc 62.90%")

Epoch 10: train loss 0.6031, train acc 67.40%, valid loss 0.6400, valid acc 64.00%
Epoch 20: train loss 0.5543, train acc 71.79%, valid loss 0.6405, valid acc 62.60%
Epoch 30: train loss 0.4943, train acc 77.27%, valid loss 0.6462, valid acc 63.70%
Early stopping: no lower validation loss in the last 10 epochs
Best epoch: 28, valid loss 0.6344, valid acc 65.30%
For comparison, all four texts (best run of the grid): valid loss 0.6479, valid acc 62.90%


## Sweep
The grid above uses exponential steps and the batch size of the course exercise. Question: **do finer learning rates, another number of hidden units or a larger batch size lower the validation loss?** A W&B sweep (course slides "Project Discussions" (Week 7), slide 57) trains all combinations automatically: learning rate 3.16e-4, 1e-4 and 3.16e-5 (the finer steps 1e-3.5 and 1e-4.5 around the best value, slide 52), 100, 300 and 600 hidden units, and batch size 32 and 128 (the course recommends the largest batch size that fits into memory, slide 51), 18 runs.

The method is `grid` (slide 44), so that starting the sweep again tests the same configurations: `random` and `bayes` (slides 45 and 46) sample new values, and the W&B documentation lists no seed for them. Every run has seed 42 and `drop_last=True`, as in `train()`; with batch size 128, it skips 33 examples per epoch (16801 = 131 x 128 + 33). As in `train()`, a run stops after 10 epochs without a lower validation loss (patience 10, course slides "Project Discussions" (Week 6), slide 38), after at most 300 epochs. Two combinations (100 and 300 hidden units, learning rate 1e-4, batch size 32) repeat earlier runs: their best epoch and validation loss must be the same, which checks that the sweep trains the same way as `train()`. The selection rule stays the same: a sweep run with the lowest validation loss of all candidate runs becomes the chosen configuration. Selecting among more runs also makes that validation loss more optimistic.

In [21]:
# the sweep: all combinations of these values (3 x 3 x 2 = 18 runs)
sweep_config = {
    "name": "model1-grid",
    "method": "grid",
    # the value in the run summary by which W&B compares the runs
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"values": [0.000316, 0.0001, 0.0000316]},
        "inner_dim": {"values": [100, 300, 600]},
        "batch_size": {"values": [32, 128]},
    },
}

# one run of the sweep: the W&B agent calls this function once per combination
def sweep_run(max_epochs=300, patience=10):
    # the agent puts the values of this combination into run.config
    run = wandb.init(project="nlp-project1-art-final")
    learning_rate = run.config["learning_rate"]
    inner_dim = run.config["inner_dim"]
    batch_size = run.config["batch_size"]
    # run name as for the other runs, and the settings that are the same in all runs
    name = f"glove-mlp-sweep-h{inner_dim}-lr{learning_rate}-b{batch_size}"
    run.name = name
    run.config.update({"model": "glove-mlp-sweep", "max_epochs": max_epochs, "patience": patience, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"
    # training batches with the batch size of this run
    sweep_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)

    # same training as in train(), with early stopping
    torch.manual_seed(42)
    model = Classifier(1200, 2, inner_dim=inner_dim)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in sweep_train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")


In [22]:
# register the sweep at W&B and train all its combinations (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config, project="nlp-project1-art-final")
wandb.agent(sweep_id, function=sweep_run)


Create sweep with ID: qdfpreu0
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/qdfpreu0
glove-mlp-sweep-h100-lr0.000316-b32: best epoch 13 of 23, valid loss 0.6445, valid acc 63.20%
glove-mlp-sweep-h100-lr0.0001-b32: best epoch 21 of 31, valid loss 0.6489, valid acc 62.60%
glove-mlp-sweep-h100-lr3.16e-05-b32: best epoch 37 of 47, valid loss 0.6504, valid acc 62.20%
glove-mlp-sweep-h300-lr0.000316-b32: best epoch 7 of 17, valid loss 0.6521, valid acc 63.00%
glove-mlp-sweep-h300-lr0.0001-b32: best epoch 14 of 24, valid loss 0.6479, valid acc 62.90%
glove-mlp-sweep-h300-lr3.16e-05-b32: best epoch 29 of 39, valid loss 0.6491, valid acc 62.40%
glove-mlp-sweep-h600-lr0.000316-b32: best epoch 6 of 16, valid loss 0.6497, valid acc 62.50%
glove-mlp-sweep-h600-lr0.0001-b32: best epoch 12 of 22, valid loss 0.6457, valid acc 62.40%
glove-mlp-sweep-h600-lr3.16e-05-b32: best epoch 22 of 32, valid loss 0.6466, valid acc 63.30%
glove-mlp-sweep

In [23]:
# load the best checkpoint of every sweep run and sort by the validation loss, as in the section "Results"
sweep_values = sweep_config["parameters"]
results = []
for learning_rate in sweep_values["learning_rate"]["values"]:
    for inner_dim in sweep_values["inner_dim"]["values"]:
        for batch_size in sweep_values["batch_size"]["values"]:
            name = f"glove-mlp-sweep-h{inner_dim}-lr{learning_rate}-b{batch_size}"
            model = Classifier(1200, 2, inner_dim=inner_dim)
            model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
            valid_loss, valid_acc = evaluate(model, valid_loader)
            results.append((valid_loss, valid_acc, name))

for valid_loss, valid_acc, name in sorted(results):
    print(f"{name}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")


glove-mlp-sweep-h100-lr0.000316-b32: valid loss 0.6445, valid acc 63.20%
glove-mlp-sweep-h600-lr0.0001-b32: valid loss 0.6457, valid acc 62.40%
glove-mlp-sweep-h600-lr0.0001-b128: valid loss 0.6459, valid acc 63.00%
glove-mlp-sweep-h600-lr3.16e-05-b32: valid loss 0.6466, valid acc 63.30%
glove-mlp-sweep-h100-lr0.000316-b128: valid loss 0.6467, valid acc 61.90%
glove-mlp-sweep-h600-lr3.16e-05-b128: valid loss 0.6475, valid acc 62.10%
glove-mlp-sweep-h300-lr0.0001-b32: valid loss 0.6479, valid acc 62.90%
glove-mlp-sweep-h300-lr0.000316-b128: valid loss 0.6488, valid acc 61.90%
glove-mlp-sweep-h300-lr3.16e-05-b128: valid loss 0.6489, valid acc 62.00%
glove-mlp-sweep-h100-lr0.0001-b32: valid loss 0.6489, valid acc 62.60%
glove-mlp-sweep-h300-lr3.16e-05-b32: valid loss 0.6491, valid acc 62.40%
glove-mlp-sweep-h300-lr0.0001-b128: valid loss 0.6491, valid acc 62.70%
glove-mlp-sweep-h100-lr0.0001-b128: valid loss 0.6495, valid acc 61.80%
glove-mlp-sweep-h600-lr0.000316-b32: valid loss 0.6497, 

The best sweep run has 100 hidden units, the smallest value of the sweep, so one more run tests the next smaller value, 50 hidden units, with the same learning rate and batch size (course slides "Project Discussions" (Week 7), slide 53). Its learning rate lies inside the tested range: 0.001 and 0.0001 are worse with 100 hidden units (0.6494 and 0.6489). Batch size 32 is the smallest value tested; smaller values are not tested, because the course recommends the largest batch size that fits into memory (slide 51).

In [24]:
# the next smaller number of hidden units, with the settings of the best sweep run
boundary_config = {
    "name": "model1-boundary",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"values": [0.000316]},
        "inner_dim": {"values": [50]},
        "batch_size": {"values": [32]},
    },
}
boundary_id = wandb.sweep(boundary_config, project="nlp-project1-art-final")
wandb.agent(boundary_id, function=sweep_run)


Create sweep with ID: 49t7nyo9
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/49t7nyo9
glove-mlp-sweep-h50-lr0.000316-b32: best epoch 7 of 17, valid loss 0.6524, valid acc 60.40%


All runs are scored on the same 1000 validation examples, so two runs can be compared example by example with the exact McNemar test (McNemar, 1947), as in `04_model2_rnn.ipynb`: the chosen run against the best run of the grid, and against the run that learned with the lowest accuracy (300 hidden units, learning rate 0.001).

In [25]:
from scipy.stats import binomtest

# True where a saved run predicts the validation example correctly
def correct_predictions(name, inner_dim):
    model = Classifier(1200, 2, inner_dim=inner_dim)
    model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
    model.eval()
    with torch.no_grad():
        return model(X_valid).argmax(dim=1) == y_valid

chosen = correct_predictions("glove-mlp-sweep-h100-lr0.000316-b32", 100)
for name in ["glove-mlp-h300-lr0.0001-e100", "glove-mlp-h300-lr0.001-e20"]:
    other = correct_predictions(name, 300)
    # & = both True, ~ = not
    only_chosen = (chosen & ~other).sum().item()
    only_other = (~chosen & other).sum().item()
    # exact McNemar test: binomial test on the examples that only one of the two runs gets right
    p = binomtest(only_chosen, only_chosen + only_other, 0.5).pvalue
    print(f"chosen vs. {name}: accuracy {chosen.float().mean().item():.2%} vs. {other.float().mean().item():.2%}, only chosen right {only_chosen}, only other right {only_other}, p = {p:.4f}")


chosen vs. glove-mlp-h300-lr0.0001-e100: accuracy 63.20% vs. 62.90%, only chosen right 61, only other right 58, p = 0.8546
chosen vs. glove-mlp-h300-lr0.001-e20: accuracy 63.20% vs. 60.20%, only chosen right 126, only other right 96, p = 0.0514


## Other seeds
All runs above use seed 42. Question: **how much do the validation scores change with the seed, and does the chosen run stay ahead of the best run of the grid?** The number of training runs is a decision of its own, and a report can give the average of several runs to show how robust a model is (course slides "Project Discussions" (Week 7), slide 42). Both configurations are trained with two more seeds, 1 and 2, otherwise as in the sweep. The seed is not tuned, because performance should not rely on it (slide 56): the chosen checkpoint stays the one with seed 42, and these runs are only reported.

In [26]:
# the two configurations: hidden units, learning rate, and the checkpoint name of the run with seed 42
seed_configurations = {
    "chosen": (100, 0.000316, "glove-mlp-sweep-h100-lr0.000316-b32"),
    "best-of-grid": (300, 0.0001, "glove-mlp-h300-lr0.0001-e100"),
}

seed_config = {
    "name": "model1-seeds",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "configuration": {"values": list(seed_configurations)},
        "seed": {"values": [1, 2]},
    },
}

# one run: as in sweep_run(), with the seed of this run
def seed_run(batch_size=32, max_epochs=300, patience=10):
    run = wandb.init(project="nlp-project1-art-final")
    configuration = run.config["configuration"]
    seed = run.config["seed"]
    inner_dim, learning_rate, _ = seed_configurations[configuration]
    name = f"glove-mlp-seed{seed}-h{inner_dim}-lr{learning_rate}"
    run.name = name
    run.config.update({"model": "glove-mlp-seeds", "inner_dim": inner_dim, "learning_rate": learning_rate, "batch_size": batch_size, "max_epochs": max_epochs, "patience": patience})
    checkpoint_path = f"checkpoints/{name}.pth"
    seed_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)

    # the seed of this run instead of 42
    torch.manual_seed(seed)
    model = Classifier(1200, 2, inner_dim=inner_dim)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in seed_train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")

seed_id = wandb.sweep(seed_config, project="nlp-project1-art-final")
wandb.agent(seed_id, function=seed_run)


Create sweep with ID: mqm0c0tz
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/mqm0c0tz
glove-mlp-seed1-h100-lr0.000316: best epoch 10 of 20, valid loss 0.6459, valid acc 63.50%
glove-mlp-seed2-h100-lr0.000316: best epoch 12 of 22, valid loss 0.6507, valid acc 62.80%
glove-mlp-seed1-h300-lr0.0001: best epoch 9 of 19, valid loss 0.6474, valid acc 62.50%
glove-mlp-seed2-h300-lr0.0001: best epoch 12 of 22, valid loss 0.6445, valid acc 63.80%


In [27]:
# validation loss and accuracy of all three seeds, from the saved checkpoints
for configuration, (inner_dim, learning_rate, seed42_name) in seed_configurations.items():
    names = [seed42_name] + [f"glove-mlp-seed{seed}-h{inner_dim}-lr{learning_rate}" for seed in [1, 2]]
    scores = []
    for name in names:
        model = Classifier(1200, 2, inner_dim=inner_dim)
        model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
        scores.append(evaluate(model, valid_loader))
    losses, accuracies = zip(*scores)
    # ddof=1: sample standard deviation of the three runs
    print(f"{configuration}: valid loss {np.mean(losses):.4f} (sd {np.std(losses, ddof=1):.4f}, seeds {[round(float(v), 4) for v in losses]}), valid acc {np.mean(accuracies):.2%} (sd {np.std(accuracies, ddof=1):.2%}, seeds {[round(v * 100, 2) for v in accuracies]})")


chosen: valid loss 0.6470 (sd 0.0033, seeds [0.6445, 0.6459, 0.6507]), valid acc 63.17% (sd 0.35%, seeds [63.2, 63.5, 62.8])
best-of-grid: valid loss 0.6466 (sd 0.0018, seeds [0.6479, 0.6474, 0.6445]), valid acc 63.07% (sd 0.67%, seeds [62.9, 62.5, 63.8])


## Conclusion
- **Chosen configuration:** 100 hidden units, learning rate 0.000316, batch size 32, best epoch 13 (from the sweep): validation loss 0.6445, accuracy 63.20%, the lowest validation loss of all 32 candidate runs.
- **All candidate runs that learned are close:** validation loss 0.6445 to 0.6554, accuracy 60.20% to 63.30%. The chosen run is not separable from the best run of the grid (exact McNemar test, p = 0.8546); the selection rule was fixed in advance.
- **Learning rate:** 0.01 can collapse training: with 50 hidden units, no ReLU unit stays active and the model predicts h1 for every example (50.80%). With 100 hidden units, 0.001 and 0.0001 are worse than 0.000316 (0.6494, 0.6489). More epochs do not help 0.01 with 100 hidden units: the run stops at epoch 30 with the same best epoch 20 (0.6524).
- **Hidden units:** with learning rate 0.000316, 50 and 300 are worse than 100 (0.6524, 0.6521).
- **Few units are enough:** with learning rate 0.001, only 5 of 50, 17 of 100 and 40 of 300 units are active on more than 1% of the examples; a single such unit still reaches 62.00%. The experiments do not show why.
- **Layer normalization:** no lower validation loss (0.6554 vs. 0.6479). One learning rate, one seed.
- **Sweep:** finer learning rates, 600 hidden units and batch size 128 change little (19 runs, 0.6445 to 0.6524). The two repeated runs reproduce their earlier results exactly.
- **Other seeds:** over seeds 42, 1 and 2, the chosen configuration (mean validation loss 0.6470, sd 0.0033; mean accuracy 63.17%, sd 0.35%) does not stay ahead of the best configuration of the grid (0.6466, sd 0.0018; 63.07%, sd 0.67%). The seed moves the validation loss at least as much as the difference between them. The chosen checkpoint stays the one with seed 42, because the seed is not tuned.
- **Input format:** with the same settings, the variant `per-choice` in `04b_preprocessing_experiments.ipynb` reaches a validation loss of 0.5829 and 69.10%, beyond the range of these runs, so the input format may matter more than the hyperparameters tuned here. It does not replace the chosen run: the selection stays within the fixed-position format, and `05_evaluation.ipynb` evaluates it as a separate comparison.
- **Hypothesis-only check:** with only h1 and h2, the best configuration of the grid reaches a lower validation loss (0.6344 vs. 0.6479) and a higher accuracy (65.30% vs. 62.90%), so in this input format the observations show no measurable benefit on the validation set. It is unfiltered and the test set adversarially filtered (Bhagavatula et al., 2020; see "Data" in `README.md`), so this may not carry over.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier`, based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Parameters and test forward pass: `named_parameters()` and a forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss: cross-entropy from the course slides "Preliminaries" (Week 1), slide 27; `criterion` and `loss = criterion(output, target)` as in the course exercise "PyTorch" (Week 1); `nn.CrossEntropyLoss` (applies the softmax to the logits) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html; the term "logits" from the course slides "Project Discussions" (Week 5), slide 27.
- Random seed: fixed seed 42 as in the course exercise "Experiment Tracking" (`seed_everything(42)`); `torch.manual_seed` from the PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html.
- Optimizer and training step: Adam from the course slides "Preliminaries" (Week 1), slide 29; `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` as in the course exercise "PyTorch" (Week 1).
- Batches: `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; `next(iter(...))` as in the PyTorch tutorial "Datasets & DataLoaders", https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html.
- Evaluation: collecting the batch losses, `argmax` predictions and labels, joining them with `torch.cat` and averaging the batch losses as in the validation of the course exercise "Experiment Tracking" (`validation_step` and `on_validation_epoch_end`); `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html; `np.mean` from NumPy (Harris et al., 2020).
- Training loop: epochs with a train loop over all batches and an evaluation after each epoch, and `model.train()`, as in the PyTorch tutorial "Optimizing Model Parameters"; the training step inside as in the course exercise "PyTorch" (Week 1); training as a function `train()` with the hyperparameters as arguments (with default values) and the seed set inside, as in the course exercise "Experiment Tracking".
- Experiment tracking: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; run name `name=` from the W&B `wandb.init()` reference, https://docs.wandb.ai/models/ref/python/functions/init; logging the metrics once per epoch with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log (written there as `run.log`, which is the same as `wandb.log` for the current run); `train_loader.batch_size` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; Weights and Biases (Biewald, 2020); `nbformat` installed for W&B, as in the course exercise "Experiment Tracking".
- Saving the best model: `torch.save(model.state_dict(), ...)` and `model.load_state_dict(torch.load(..., weights_only=True))` as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html; `os.makedirs` from the Python standard library, https://docs.python.org/3/library/os.html; `float("inf")` from the Python documentation, https://docs.python.org/3/library/functions.html#float; best values in the run summary (`run.summary[...]`, here `wandb.run.summary` for the current run) as in the W&B documentation "Log summary metrics", https://docs.wandb.ai/models/track/log/log-summary.
- Experiments: hyperparameter tuning on the validation set, as on slide 10 of the course slides "Course Project HS26" (compare model settings, select the best model and hyperparameters).
- Results: loading the checkpoints with `load_state_dict` as in the PyTorch tutorial "Save and Load the Model"; sorting the results with `sorted()` (tuples are compared by their first item first) as in the Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html.
- Active ReLU units: term and explanation "dying ReLU" from Lu, Shin, Su & Karniadakis (2020), *Dying ReLU and Initialization: Theory and Numerical Examples*, Communications in Computational Physics, https://doi.org/10.4208/cicp.OA-2020-0165; `torch.no_grad()` as in the PyTorch tutorial "Optimizing Model Parameters"; share of examples per unit with `.float().mean(dim=0)` as in the evaluation (`torch.mean`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.mean.html); counting the active units with `.sum().item()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters".
- Training decisions: course slides "Project Discussions": 1 output with a sigmoid or 2 outputs with a softmax (Week 5, slide 22), "double sigmoid" (Week 6, slide 30), Adam and AdamW (Week 6, slide 31), the learning rate as the most important hyperparameter (Week 6, slide 32), a constant learning rate (Week 6, slide 33, citing Hägele, A., Bakouch, E., Kosson, A., Ben Allal, L., Von Werra, L., & Jaggi, M. (2024), *Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations*, https://arxiv.org/abs/2405.18392), regularization (Week 6, slide 34), early stopping (Week 6, slide 38), batch size (Week 4, slide 14). AdamW: algorithm in the PyTorch documentation `torch.optim.AdamW`, https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html, from Loshchilov & Hutter (2019), *Decoupled Weight Decay Regularization*, ICLR 2019, https://arxiv.org/abs/1711.05101.
- Grid search, learning rate 0.0001, learning rate 0.00001 and 1000 hidden units: grid search (all combinations) from the course slides "Project Discussions" (Week 7), slide 44; testing beyond the boundary of the tested range (slide 53), exponentially changing values (slide 52) and a sensible number of epochs (slide 41) from the same slides.
- Hypothesis-only check: hypothesis-only baselines from Gururangan, S., Swayamdipta, S., Levy, O., Schwartz, R., Bowman, S., & Smith, N. A. (2018), *Annotation Artifacts in Natural Language Inference Data*, NAACL 2018, https://arxiv.org/abs/1803.02324, and Poliak, A., Naradowsky, J., Haldar, A., Rudinger, R., & Van Durme, B. (2018), *Hypothesis Only Baselines in Natural Language Inference*, *SEM 2018, https://arxiv.org/abs/1805.01042; for ART: Bhagavatula et al. (2020), Table 1 (BERT-ft [h_i Only], 59.5%); column indexing `[:, ...]` (here `X_train[:, 600:]`) as in the course exercise "PyTorch" (Week 1); training steps as in `train()` above; origin of the splits: Bhagavatula et al. (2020), Table 6, and course slides "Course Project HS26", slide 8.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md; all code reviewed and understood by me.
- Layer normalization: `nn.LayerNorm` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html; placed between the linear layer and the ReLU following Ba, Kiros & Hinton (2016), *Layer Normalization* ("after the normalization but before the non-linearity"), https://arxiv.org/abs/1607.06450; normalization as an option and layer norm (yes/no) as an experiment from the course slides "Project Discussions" (Week 6), slide 34, and (Week 7), slide 44. Check: `torch.equal`, https://docs.pytorch.org/docs/stable/generated/torch.equal.html, and `torch.numel`, https://docs.pytorch.org/docs/stable/generated/torch.numel.html, from the PyTorch documentation.
- Sweep: W&B sweeps from the course slides "Project Discussions" (Week 7), slide 57, and the W&B documentation on the sweep configuration, https://docs.coreweave.com/models/sweeps/define-sweep-configuration, its options (`method`, `metric`, `parameters`, `values`), https://docs.coreweave.com/models/sweeps/sweep-config-keys, and the sweeps walkthrough (`wandb.sweep`, `wandb.agent` with `function`, `wandb.init` inside the function, `run.config`), https://docs.coreweave.com/models/sweeps/walkthrough; `run.config.update`, W&B documentation on the run config, https://docs.coreweave.com/models/track/config; `run.name`, W&B documentation `Run`, https://docs.coreweave.com/models/ref/python/experiments/run; grid search (slide 44), random search (slide 45), Bayesian optimization (slide 46), finer steps such as 1e-4.5 (slide 52), the next value beyond the boundary of the tested range (slide 53) and the batch size (slide 51) from the same slides; early stopping with a patience from the same slides (Week 6), slide 38. Comparison of two runs: exact McNemar test (McNemar, 1947) with `scipy.stats.binomtest` as in `04_model2_rnn.ipynb` (see the sources there), https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html.
- Other seeds: reporting the average of several runs and the debate about tuning the seed from the course slides "Project Discussions" (Week 7), slides 42 and 56; `np.std` with `ddof`, NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.std.html; `zip(*scores)` and `round()`, Python documentation, https://docs.python.org/3/library/functions.html; `list()` of a dictionary (its keys), Python tutorial "Dictionaries", https://docs.python.org/3/tutorial/datastructures.html#dictionaries.
- Output of W&B: `WANDB_SILENT` (no W&B log lines in the notebook output), W&B documentation "Environment variables", https://docs.coreweave.com/models/track/environment-variables; `os.environ`, Python documentation, https://docs.python.org/3/library/os.html#os.environ.